# 03 — Búsqueda guiada: evidencia histórica y estimaciones

La búsqueda A/B es una aplicación secundaria exploratoria, separada del benchmark principal por regresor. A proyecta propuestas a diseños históricos y obtiene respuestas registradas después de actuar. B consulta un sustituto congelado para configuraciones nuevas: esas respuestas son estimaciones pendientes de FEM.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = {"png", "svg"}
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from truss_research.evaluacion.presentacion import notebook_context

context = notebook_context()
CONFIG = context.config
STUDY_DIR, REPORT_DIR = context.study_dir, context.report_dir
DATA_DIR = CONFIG.data_dir
display(
    Markdown(
        f"**Estudio:** `{STUDY_DIR.name}` — **etiqueta:** `{CONFIG.label}` — **informe:** `{REPORT_DIR.name}`"
    )
)
if CONFIG.label == "comprobacion":
    display(
        Markdown(
            "**Comprobación técnica: sus métricas no constituyen resultados de tesis.**"
        )
    )

## 1. Estrategias y presupuesto efectivos

La configuración define las estrategias participantes, población inicial y presupuestos; no se trasladan los valores del catálogo anterior. La población inicial consume presupuesto. Las políticas se entrenan en mundos definidos con ajuste; validación selecciona checkpoints sin actualizar sus gradientes. Prueba se consulta tras congelar decisiones.

Evolución genética, evolución diferencial, aleatorio, greedy, Double DQN y bandido contextual son comparadores con funciones distintas. Las variantes asistidas usan un predictor auxiliar y registran su costo.

In [ ]:
display(
    pd.Series(
        {
            "estrategias": CONFIG.search.strategies,
            "presupuestos": CONFIG.budgets,
            "población_inicial": CONFIG.evolution.population_size,
            "propuestas_por_lote": CONFIG.proposals.pool_size,
            "semillas": CONFIG.seeds,
            "escenarios_pareados": CONFIG.proposals.evaluation_episodes,
            "interacciones_por_política": CONFIG.rl_steps,
        }
    )
)
comparison = pd.read_csv(REPORT_DIR / "comparison.csv")
display(comparison)

## 2. A: recuperación retrospectiva

Las respuestas de A corresponden a los diseños registrados a los que se proyectan las propuestas. Las distancias y consultas sin topología compatible delimitan la interpretación. Una consulta sin soporte consume presupuesto. Menos consultas en este banco no prueban ahorro FEM.

Se comparan estrategias dentro del mismo experimento, partición y presupuesto. El resultado no valida la geometría propuesta antes de proyectarla.

In [ ]:
historical = comparison.loc[comparison["experiment"].eq("A")]
display(historical)
for split, label in (("validation", "Validación"), ("test", "Prueba")):
    historical.loc[historical["split"].eq(split)].pivot(
        index="budget", columns="strategy", values="mean_auc"
    ).plot.bar(
        figsize=(12, 5), ylabel="AUC", title=f"A: respuestas históricas · {label}"
    )
    plt.tight_layout()
    plt.show()

## 3. B: generación evaluada por un sustituto

B usa respuestas predictivas y soporte definido con ajuste. Acuerdo entre miembros del evaluador no constituye incertidumbre física calibrada. Su HV/AUC describe calidad estimada, no respuestas FEM nuevas; las magnitudes de A y B no se interpretan como evidencia intercambiable.

In [ ]:
generative = comparison.loc[comparison["experiment"].eq("B")]
display(generative)
for split, label in (("validation", "Validación"), ("test", "Prueba")):
    generative.loc[generative["split"].eq(split)].pivot(
        index="budget", columns="strategy", values="mean_auc"
    ).plot.bar(
        figsize=(12, 5), ylabel="AUC", title=f"B: estimaciones predictivas · {label}"
    )
    plt.tight_layout()
    plt.show()

## 4. Terminación y alcance

Se conservan episodios que agotan soporte o presupuesto. El alcance usa una referencia de entrenamiento congelada, no etiquetas finales para fijar el umbral. Las consultas que no alcanzan el objetivo siguen censuradas; no se sustituyen por éxitos.

In [ ]:
display(pd.read_csv(REPORT_DIR / "termination_counts.csv"))
display(pd.read_csv(REPORT_DIR / "target_attainment.csv"))

## 5. Costos y conclusiones admisibles

Se separa entrenamiento de políticas de tiempo por búsqueda, generación, selección, evaluación y asistencia. La amortización exportada es exploratoria, compara costos de políticas y excluye predictores compartidos y FEM. RL puede ser superado por los controles sin invalidar el estudio.

In [ ]:
display(pd.read_csv(REPORT_DIR / "policy_costs.csv"))
amortization = pd.read_csv(REPORT_DIR / "amortization.csv")
if amortization.empty:
    display(
        Markdown(
            "No hay comparaciones de amortización disponibles para estas estrategias."
        )
    )
else:
    display(amortization)

## 6. Curvas de calidad y frentes finales

Las curvas usan los episodios exportados y prolongan como meseta los episodios terminados. Los frentes muestran el escenario 0 y la semilla representativa al mayor presupuesto. Son diagnósticos de una ejecución, no intervalos poblacionales. A conserva respuestas registradas y B estimaciones; no se certifican diseños nuevos.

In [ ]:
from truss_research.evaluacion.figuras import search_figures

for figure in search_figures(REPORT_DIR, max(CONFIG.budgets)):
    display(figure)
    plt.close(figure)